### LLM AS A JUDGE

In [2]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [4]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [5]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [8]:
rec = answers[0]
rec

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late. If you want to receive a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [9]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [10]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the original meaning: late enrollment is allowed, and certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent to the ground truth.', score='good')

In [11]:
calc_price(usage)

{'input_cost': 0.00022275000000000002,
 'output_cost': 0.0002295,
 'total_cost': 0.00045225}

In [12]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [13]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joiners can still participate, and certificate eligibility requires submitting the project before submissions close. This is semantically equivalent.', score='good')

In [14]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [15]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/575 [00:00<?, ?it/s]

In [16]:
results[10]

({'question': 'How do students join the Office Hours or live workshop if the Zoom link isn’t public?',
  'document': '489dd1c9d9',
  'score': 'good',
  'reasoning': 'The AI answer preserves all key points from the ground truth: the Zoom link is only for instructors/presenters/TAs, students watch via YouTube Live, the video URL is posted in Telegram/Slack announcements, questions are submitted through Slido, and chat questions may be missed. This is semantically equivalent.'},
 ResponseUsage(input_tokens=447, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=79, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=526))

In [17]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [18]:
df_eval = pd.DataFrame(evaluations)

In [19]:
df_eval.head()

,question,document,score,reasoning
0,Can I still join the course if I found it late?,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Am I allowed to enroll after the course has al...,74eb249bbf,bad,"The ground truth says yes, late enrollment is ..."
2,"If I join now, can I still get a certificate?",74eb249bbf,bad,The AI answer preserves the key point that a c...
3,What do I need to do to qualify for the certif...,74eb249bbf,good,The AI answer preserves the key point that lat...
4,"Is it too late to participate, or can I still ...",74eb249bbf,good,The AI answer conveys the core idea that parti...


In [20]:
calc_total_price(usages)

0.40427325

In [21]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 548/575 = 95.30%


In [22]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
1,Am I allowed to enroll after the course has al...,74eb249bbf,bad,"The ground truth says yes, late enrollment is ..."
2,"If I join now, can I still get a certificate?",74eb249bbf,bad,The AI answer preserves the key point that a c...
14,Should I ask questions in the chat during Offi...,489dd1c9d9,bad,The AI answer captures the key point: do not a...
33,What do I have to pass in order to receive the...,9f689c185f,bad,The ground truth says the certificate requires...
41,Do you know the next offering date for the cou...,bd31146b0e,bad,The ground truth gives a specific next offerin...


In [27]:
df_eval.score.value_counts(normalize=True)

score
good    0.953043
bad     0.046957
Name: proportion, dtype: float64

In [28]:
evaluations[1]

{'question': 'Am I allowed to enroll after the course has already started?',
 'document': '74eb249bbf',
 'score': 'bad',
 'reasoning': "The ground truth says yes, late enrollment is allowed, with the condition that to receive a certificate the project must be submitted while submissions are still being accepted. The AI answer says 'I don't know,' which does not convey the correct answer and misses the key information."}

A practical approach is to build a simple application with Streamlit. Show each question, the original answer, the generated answer, and the judge verdict side by side. Then mark each verdict as correct or incorrect and use that feedback to adjust the judge instructions. This is a lot of trial and error, but it makes the evaluation framework more reliable.

In [29]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

All the files are generated for the course materials on 16-07-2026. The run used 575 RAG answers.

In [30]:
good_count = (df_eval["score"] == "good").sum()
bad_count = (df_eval["score"] == "bad").sum()
print(f"Good: {good_count}")
print(f"Bad: {bad_count}")

Good: 548
Bad: 27


In [31]:
calc_total_price(usages)

0.40427325